# SPIN Baseline Comparison

Faithful reimplementation of SPIN (Sarkar et al., EMNLP 2025) using `attentionSPIN.py` logic.

**Pipeline:**
1. Load the same 400-image eval set as Stage 4
2. Generate **Baseline (greedy)** and **SPIN** captions at fixed token budgets
3. Score with CHAIR + bootstrap 95% CIs

**Token budgets:** 80, 64, 128 (`max_new_tokens`)

**SPIN config (paper Table 1, LLaVA-7B greedy CHAIR):**
- `routed_head = 0.95` — keep top 95% of heads (r=0.05 suppressed)
- `small_num_mask = 0.08` — scale suppressed heads to α=0.08
- `start_layer = 0, end_layer = 32`

**Colab:** A100 GPU · ~25–30 min per 400-image budget · resumable checkpoints in Drive

Experiment log: `results/spin_experiment_manifest.json`


## 0. Install (run once, restart runtime, then skip)

In [1]:
# import os
# os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
# !pip install -q 'numpy==1.26.4'
# !pip install -q 'transformers>=4.47' 'accelerate>=0.33' 'tokenizers>=0.21'
# !pip install -q peft bitsandbytes safetensors 'torchao>=0.16.0'
# !pip install -q pillow tqdm spacy sentencepiece
# !python -m spacy download en_core_web_sm -q
# print('Done. Runtime -> Restart session, then skip this cell.')

## 1. Imports + GPU check

In [2]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import json, gc, time, math, types, urllib.request
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu':
    raise RuntimeError('No GPU — Runtime -> Change runtime type -> A100 GPU')
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/llava_hallucination_heads')
DRIVE.mkdir(parents=True, exist_ok=True)
(DRIVE / 'cache').mkdir(exist_ok=True)
(DRIVE / 'results').mkdir(exist_ok=True)

LOCAL = Path('/content/spin_work')
LOCAL.mkdir(exist_ok=True)
(LOCAL / 'images').mkdir(exist_ok=True)

GPU:  NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB
Mounted at /content/drive


## 2. Download Stage 4 results from GitHub

In [3]:
S4_URL   = 'https://raw.githubusercontent.com/armaansandhu26/causal-grounding-lora/refs/heads/master/results/stage4_400img_results.json'
S4_LOCAL = LOCAL / 'stage4_400img_results.json'

if not S4_LOCAL.exists():
    print('Downloading Stage 4 results from GitHub...')
    urllib.request.urlretrieve(S4_URL, str(S4_LOCAL))

with open(S4_LOCAL) as f:
    s4 = json.load(f)

eval_images     = [r['img_id']  for r in s4['eval_captions']]
eval_gt_objects = [set(r['gt']) for r in s4['eval_captions']]
baseline_by_id  = {r['img_id']: r['captions']['baseline']
                   for r in s4['eval_captions']}

print(f'Images: {len(eval_images)}')
print(f'Baseline captions: {len(baseline_by_id)}')
print(f'\nStage 4 CHAIR results (from repo, for reference):')
for cond, v in s4['chair'].items():
    print(f'  {cond:10s}  CHAIRs={v["CHAIRs"]:.4f}  CHAIRi={v["CHAIRi"]:.4f}')

Images: 400
Baseline captions: 400

Stage 4 CHAIR results (from repo, for reference):
  baseline    CHAIRs=0.3700  CHAIRi=0.1558
  stage2      CHAIRs=0.2650  CHAIRi=0.1043
  stage3      CHAIRs=0.3100  CHAIRi=0.1407
  stage4      CHAIRs=0.2300  CHAIRi=0.0958


## 3. Download the 400 eval images

In [4]:
IMG_DIR = LOCAL / 'images'
img_id_to_path = {}
to_dl = []

for img_id in eval_images:
    fname = f'COCO_val2014_{img_id:012d}.jpg'
    p = IMG_DIR / fname
    img_id_to_path[img_id] = str(p)
    if not p.exists():
        to_dl.append((img_id, fname, p))

if to_dl:
    print(f'Downloading {len(to_dl)} images...')
    for img_id, fname, p in tqdm(to_dl, desc='Images'):
        try:
            urllib.request.urlretrieve(
                f'http://images.cocodataset.org/val2014/{fname}', str(p))
        except Exception as e:
            print(f'  Failed {img_id}: {e}')
else:
    print('All images already downloaded.')

print(f'Ready: {len(eval_images)} images')

Images:   0%|          | 0/400 [00:00<?, ?it/s]

Ready: 400 images


## 4. Load LLaVA-1.5-7B (base model only, no LoRA)

In [5]:
from transformers import AutoProcessor, LlavaForConditionalGeneration

MODEL_ID  = 'llava-hf/llava-1.5-7b-hf'
processor = AutoProcessor.from_pretrained(MODEL_ID)

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    attn_implementation='eager',
    device_map={'': 0},
)
model.eval()

text_cfg       = model.config.text_config
NUM_LAYERS     = text_cfg.num_hidden_layers
NUM_HEADS      = text_cfg.num_attention_heads
HEAD_DIM       = text_cfg.hidden_size // NUM_HEADS
IMAGE_TOKEN_ID = model.config.image_token_index
vision_cfg     = model.config.vision_config
NUM_IMG_TOKENS = (vision_cfg.image_size // vision_cfg.patch_size) ** 2
PROMPT         = 'USER: <image>\nDescribe this image in detail.\nASSISTANT:'

print(f'Model loaded. Layers={NUM_LAYERS}, Heads={NUM_HEADS}, HeadDim={HEAD_DIM}')
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/70.1k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Model loaded. Layers=32, Heads=32, HeadDim=128
VRAM: 14.13 GB


## 5. SPIN generation (canonical implementation)

Port of `attentionSPIN.py` with transformers compatibility fixes (GQA, `position_embeddings`, 2-value return).
**Use this cell only** — the next markdown cell replaces an older broken version.


> **Removed:** An older SPIN implementation in this slot failed on current `transformers` (rotary API / return arity). Section 5 above is the working version.


In [6]:
# ── Shared SPIN hyperparameters (match manifest) ──
SPIN_START_LAYER    = 0
SPIN_END_LAYER      = 32
SPIN_ROUTED_HEAD    = 0.95
SPIN_SMALL_NUM_MASK = 0.08
TOKEN_BUDGETS       = [80, 64, 128]
BUDGETS_ALL         = [80, 64, 128]

from transformers.models.llama.modeling_llama import apply_rotary_pos_emb
import types, math
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image


def get_visual_token_span(input_ids):
    ids = input_ids[0]
    mask = (ids == IMAGE_TOKEN_ID)
    pos = mask.nonzero(as_tuple=True)[0]
    n_ph = int(mask.sum().item())

    if n_ph == 0:
        raise ValueError("No IMAGE_TOKEN_ID found in input_ids.")

    if n_ph >= NUM_IMG_TOKENS:
        return int(pos[0].item()), int(pos[-1].item()) + 1

    return int(pos[0].item()), int(pos[0].item()) + NUM_IMG_TOKENS


def repeat_kv(hidden_states, n_rep):
    """
    hidden_states: [bsz, num_kv_heads, seq_len, head_dim]
    returns:       [bsz, num_heads, seq_len, head_dim]
    """
    if n_rep == 1:
        return hidden_states

    bsz, num_kv_heads, slen, head_dim = hidden_states.shape
    hidden_states = hidden_states[:, :, None, :, :].expand(
        bsz, num_kv_heads, n_rep, slen, head_dim
    )
    return hidden_states.reshape(bsz, num_kv_heads * n_rep, slen, head_dim)


def llama_spin_forward(
    self,
    hidden_states,
    attention_mask=None,
    position_ids=None,
    past_key_value=None,
    output_attentions=False,
    use_cache=False,
    cache_position=None,
    position_embeddings=None,
    **kwargs,
):
    # Some newer transformers versions pass this as past_key_values
    if past_key_value is None and "past_key_values" in kwargs:
        past_key_value = kwargs["past_key_values"]

    bsz, q_len, _ = hidden_states.size()

    num_heads = self.config.num_attention_heads
    num_kv_heads = getattr(self.config, "num_key_value_heads", num_heads)
    num_kv_groups = num_heads // num_kv_heads

    head_dim = self.config.hidden_size // num_heads
    hidden_size = self.config.hidden_size

    query_states = self.q_proj(hidden_states)
    key_states = self.k_proj(hidden_states)
    value_states = self.v_proj(hidden_states)

    query_states = query_states.view(
        bsz, q_len, num_heads, head_dim
    ).transpose(1, 2)

    key_states = key_states.view(
        bsz, q_len, num_kv_heads, head_dim
    ).transpose(1, 2)

    value_states = value_states.view(
        bsz, q_len, num_kv_heads, head_dim
    ).transpose(1, 2)

    # New transformers path: decoder layer passes cos/sin directly
    if position_embeddings is not None:
        cos, sin = position_embeddings

    # Old transformers fallback
    elif hasattr(self, "rotary_emb"):
        kv_seq_len = key_states.shape[-2]

        if past_key_value is not None:
            try:
                kv_seq_len += past_key_value.get_usable_length(kv_seq_len, self.layer_idx)
            except Exception:
                pass

        cos, sin = self.rotary_emb(value_states, seq_len=kv_seq_len)

    else:
        raise RuntimeError(
            "No self.rotary_emb and no position_embeddings received. "
            "This is a transformers/LlamaAttention API mismatch."
        )

    # Your transformers version expects:
    # apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1)
    # So do NOT pass position_ids here.
    query_states, key_states = apply_rotary_pos_emb(
        query_states,
        key_states,
        cos,
        sin,
    )

    if past_key_value is not None:
        cache_kwargs = {
            "sin": sin,
            "cos": cos,
            "cache_position": cache_position,
        }

        try:
            key_states, value_states = past_key_value.update(
                key_states,
                value_states,
                self.layer_idx,
                cache_kwargs,
            )
        except TypeError:
            key_states, value_states = past_key_value.update(
                key_states,
                value_states,
                self.layer_idx,
            )

    key_states = repeat_kv(key_states, num_kv_groups)
    value_states = repeat_kv(value_states, num_kv_groups)

    attn_weights = torch.matmul(
        query_states,
        key_states.transpose(2, 3),
    ) / math.sqrt(head_dim)

    if attention_mask is not None:
        causal_mask = attention_mask[:, :, :, : key_states.shape[-2]]
        attn_weights = attn_weights + causal_mask

    # ── SPIN gating ──
    num_routed_head = max(1, int(self.routed_head * num_heads))

    img_start = int(self.img_start_idx)
    img_end = min(int(self.img_end_idx), attn_weights.shape[-1])

    attn_scores = attn_weights.permute(0, 2, 1, 3)

    attn_scores_headwise = attn_scores[:, -1, :, img_start:img_end].sum(dim=-1)
    attn_scores_headwise = attn_scores_headwise.view(-1, num_heads)

    attn_score_std = attn_scores_headwise.std(dim=1, keepdim=True)
    attn_score_norm = attn_scores_headwise / (attn_score_std + 1e-8)

    gates = F.softmax(attn_score_norm, dim=1)

    _, indices = torch.topk(gates, k=num_routed_head, dim=1)

    mask = F.one_hot(indices, num_classes=num_heads).sum(dim=1).to(query_states.dtype)
    mask[mask == 0] = self.small_num_mask

    if q_len > 1:
        prefix_mask = torch.ones(
            (bsz * (q_len - 1), num_heads),
            dtype=query_states.dtype,
            device=query_states.device,
        )
        mask = torch.cat([prefix_mask, mask], dim=0)

    mask = mask.reshape(bsz, q_len, num_heads)
    # ─────────────────

    attn_weights = nn.functional.softmax(
        attn_weights,
        dim=-1,
        dtype=torch.float32,
    ).to(query_states.dtype)

    attn_output = torch.matmul(attn_weights, value_states)

    attn_output = attn_output.transpose(1, 2).reshape(
        bsz,
        q_len,
        num_heads,
        head_dim,
    )

    attn_output = torch.einsum(
        "bnh,bnhd->bnhd",
        mask,
        attn_output,
    )

    attn_output = attn_output.reshape(bsz, q_len, hidden_size)
    attn_output = self.o_proj(attn_output)

    if not output_attentions:
        attn_weights = None

    # IMPORTANT:
    # Your LlamaDecoderLayer does:
    # hidden_states, _ = self.self_attn(...)
    # So we must return exactly 2 values.
    return attn_output, attn_weights


@torch.no_grad()
def gen_spin(
    model_obj,
    image_path,
    start_layer=SPIN_START_LAYER,
    end_layer=SPIN_END_LAYER,
    routed_head=SPIN_ROUTED_HEAD,
    small_num_mask=SPIN_SMALL_NUM_MASK,
    max_new_tokens=80,
):
    """
    SPIN decoding.
    routed_head=0.95     -> keep top 95% of heads (paper r=0.05)
    small_num_mask=0.08  -> scale suppressed heads to 0.08 (paper α)
    """

    img = Image.open(image_path).convert("RGB")

    inputs = processor(
        text=PROMPT,
        images=img,
        return_tensors="pt",
    ).to(device, torch.float16)

    inputs["input_ids"] = inputs["input_ids"].long()
    inputs["attention_mask"] = inputs["attention_mask"].long()

    img_start, img_end = get_visual_token_span(inputs["input_ids"])

    layers = model_obj.model.language_model.layers
    patched_layers = range(start_layer, min(end_layer, len(layers)))

    # Patch self_attn forward on each decoder layer
    for i in patched_layers:
        attn = layers[i].self_attn

        if not hasattr(attn, "_spin_original_forward"):
            attn._spin_original_forward = attn.forward

        attn.img_start_idx = img_start
        attn.img_end_idx = img_end
        attn.routed_head = routed_head
        attn.small_num_mask = small_num_mask

        attn.forward = types.MethodType(llama_spin_forward, attn)

    eos_id = processor.tokenizer.eos_token_id

    past_kv = None
    cur_ids = inputs["input_ids"]
    cur_msk = inputs["attention_mask"]

    generated = []

    try:
        for _ in range(max_new_tokens):
            kw = dict(
                input_ids=cur_ids,
                attention_mask=cur_msk,
                use_cache=True,
                past_key_values=past_kv,
                return_dict=True,
            )

            if past_kv is None:
                kw["pixel_values"] = inputs["pixel_values"]

            out = model_obj(**kw)

            next_id = int(out.logits[:, -1, :].float().argmax(dim=-1).item())
            generated.append(next_id)

            if next_id == eos_id:
                break

            past_kv = out.past_key_values

            cur_ids = torch.tensor(
                [[next_id]],
                dtype=torch.long,
                device=device,
            )

            cur_msk = torch.cat(
                [
                    cur_msk,
                    torch.ones(
                        1,
                        1,
                        dtype=torch.long,
                        device=device,
                    ),
                ],
                dim=1,
            )

    finally:
        # Restore original forward
        for i in patched_layers:
            attn = layers[i].self_attn

            for attr in [
                "img_start_idx",
                "img_end_idx",
                "routed_head",
                "small_num_mask",
            ]:
                if hasattr(attn, attr):
                    delattr(attn, attr)

            if hasattr(attn, "_spin_original_forward"):
                attn.forward = attn._spin_original_forward
                delattr(attn, "_spin_original_forward")

    return processor.tokenizer.decode(generated, skip_special_tokens=True)


# Sanity test
test_path = img_id_to_path[eval_images[0]]

print("Sanity test...")

cap = gen_spin(model, test_path)

torch.cuda.empty_cache()

print(f"  SPIN:     {cap[:120]}")
print(f"  Baseline: {baseline_by_id[eval_images[0]][:120]}")
print(f"VRAM after test: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print("OK.")

Sanity test...
  SPIN:     The image features a yellow fire hydrant prominently placed on a sidewalk in front of a building. The fire hydrant is si
  Baseline: The scene features a yellow fire hydrant sitting on the sidewalk next to a building, possibly an old storefront or shop 
VRAM after test: 14.14 GB
OK.


## 6. Generate captions at fixed token budgets

For each budget in `{80, 64, 128}`:
- **Baseline:** greedy decode (`model.generate`)
- **SPIN:** `gen_spin(..., max_new_tokens=budget)`

Checkpoints (resume-safe, save every 10 images):
- `Drive/cache/baseline_captions_budget{b}.json`
- `Drive/cache/spin_captions_budget{b}.json`

> Delete old checkpoints from the superseded 0.8/0.1 run before re-running (`spin_captions_s4ids.json`, etc.).


In [7]:
@torch.no_grad()
def gen_baseline(model_obj, image_path, max_new_tokens=80):
    """Greedy baseline — same prompt/decoding as Stage 4."""
    img = Image.open(image_path).convert('RGB')
    inputs = processor(text=PROMPT, images=img, return_tensors='pt').to(device, torch.float16)
    inputs['input_ids'] = inputs['input_ids'].long()
    inputs['attention_mask'] = inputs['attention_mask'].long()
    out = model_obj.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=False, use_cache=True,
    )
    gen_ids = out[0, inputs['input_ids'].shape[1]:]
    return processor.tokenizer.decode(gen_ids, skip_special_tokens=True)


def _load_ckpt(path):
    if path.exists():
        with open(path) as f:
            rows = json.load(f)
        done = {int(r['img_id']) for r in rows}
        return rows, done
    return [], set()


def _save_ckpt(path, rows):
    with open(path, 'w') as f:
        json.dump(rows, f)

print('Baseline generator ready.')


Baseline generator ready.


In [ ]:
for budget in TOKEN_BUDGETS:
    spin_path = DRIVE / 'cache' / f'spin_captions_budget{budget}.json'
    base_path = DRIVE / 'cache' / f'baseline_captions_budget{budget}.json'

    print(f'\n{"="*60}')
    print(f'Token budget = {budget}')
    print(f'{"="*60}')

    # ── Baseline greedy ──
    base_rows, base_done = _load_ckpt(base_path)
    print(f'Baseline: {len(base_done)}/{len(eval_images)} done')
    t0 = time.time()
    base_fail = 0
    for img_id, gt_set in tqdm(zip(eval_images, eval_gt_objects),
                               total=len(eval_images), desc=f'Baseline@{budget}'):
        if img_id in base_done:
            continue
        caption = ''
        try:
            caption = gen_baseline(model, img_id_to_path[img_id], max_new_tokens=budget)
        except Exception as e:
            base_fail += 1
            print(f'  baseline {img_id}: {e}')
            torch.cuda.empty_cache(); gc.collect()
        base_rows.append({'img_id': img_id, 'gt': list(gt_set), 'caption': caption})
        base_done.add(img_id)
        if len(base_rows) % 10 == 0:
            _save_ckpt(base_path, base_rows)
            torch.cuda.empty_cache()
    _save_ckpt(base_path, base_rows)
    print(f'Baseline@{budget}: {len(base_rows)} captions, {base_fail} failures, '
          f'{(time.time()-t0)/60:.1f} min → {base_path}')

    # ── SPIN ──
    spin_rows, spin_done = _load_ckpt(spin_path)
    print(f'SPIN: {len(spin_done)}/{len(eval_images)} done')
    t0 = time.time()
    spin_fail = 0
    for img_id, gt_set in tqdm(zip(eval_images, eval_gt_objects),
                               total=len(eval_images), desc=f'SPIN@{budget}'):
        if img_id in spin_done:
            continue
        caption = ''
        try:
            caption = gen_spin(
                model, img_id_to_path[img_id],
                start_layer=SPIN_START_LAYER,
                end_layer=SPIN_END_LAYER,
                routed_head=SPIN_ROUTED_HEAD,
                small_num_mask=SPIN_SMALL_NUM_MASK,
                max_new_tokens=budget,
            )
        except Exception as e:
            spin_fail += 1
            print(f'  spin {img_id}: {e}')
            torch.cuda.empty_cache(); gc.collect()
        spin_rows.append({'img_id': img_id, 'gt': list(gt_set), 'caption': caption})
        spin_done.add(img_id)
        if len(spin_rows) % 10 == 0:
            _save_ckpt(spin_path, spin_rows)
            torch.cuda.empty_cache()
    _save_ckpt(spin_path, spin_rows)
    print(f'SPIN@{budget}: {len(spin_rows)} captions, {spin_fail} failures, '
          f'{(time.time()-t0)/60:.1f} min → {spin_path}')

print('\nAll pending budgets complete. Copy cache/*.json to repo results/ when done.')



Token budget = 80
Baseline: 0/400 done


Baseline@80:   0%|          | 0/400 [00:00<?, ?it/s]

Baseline@80: 400 captions, 0 failures, 21.6 min → /content/drive/MyDrive/llava_hallucination_heads/cache/baseline_captions_budget80.json
SPIN: 0/400 done


SPIN@80:   0%|          | 0/400 [00:00<?, ?it/s]

SPIN@80: 400 captions, 0 failures, 27.2 min → /content/drive/MyDrive/llava_hallucination_heads/cache/spin_captions_budget80.json

Token budget = 64
Baseline: 0/400 done


...

SPIN@64:   0%|          | 0/400 [00:00<?, ?it/s]

SPIN@64: 400 captions, 0 failures, 22.2 min → /content/drive/MyDrive/llava_hallucination_heads/cache/spin_captions_budget64.json

Token budget = 128
Baseline: 0/400 done


Baseline@128:   0%|          | 0/400 [00:00<?, ?it/s]

Baseline@128: 400 captions, 0 failures, 29.7 min → /content/drive/MyDrive/llava_hallucination_heads/cache/baseline_captions_budget128.json
SPIN: 0/400 done


SPIN@128:   0%|          | 0/400 [00:00<?, ?it/s]

---
## Phase 2: Evaluation (no GPU needed)
---

## 7. COCO vocab + CHAIR scorer

In [12]:
import spacy
nlp = spacy.load('en_core_web_sm')

COCO_SYNONYMS = {
    'person':       ['man','woman','people','boy','girl','child','guy','lady',
                     'kid','baby','player','rider','skier','surfer','snowboarder'],
    'car':          ['vehicle','automobile','sedan','suv'],
    'dog':          ['puppy','dogs'],
    'cat':          ['kitten','cats'],
    'tv':           ['television','monitor','screen'],
    'couch':        ['sofa'],
    'cell phone':   ['phone','cellphone','smartphone'],
    'dining table': ['table','desk'],
    'wine glass':   ['glass'],
    'bicycle':      ['bike'],
    'motorcycle':   ['motorbike'],
    'airplane':     ['plane','jet'],
    'potted plant': ['plant'],
    'laptop':       ['computer'],
    'refrigerator': ['fridge'],
    'truck':        ['lorry'],
    'boat':         ['ship','sailboat'],
    'fire hydrant': ['hydrant'],
    'hot dog':      ['hotdog'],
    'traffic light':['stoplight'],
    'sports ball':  ['ball','football','soccer ball','basketball'],
    'baseball bat': ['bat'],
    'tennis racket':['racket','racquet'],
}
MULTIWORD = {
    'hydrant':  'fire hydrant',
    'hotdog':   'hot dog',
    'stoplight':'traffic light',
    'bat':      'baseball bat',
    'racket':   'tennis racket',
    'racquet':  'tennis racket',
}
ALL_COCO = [
    'person','bicycle','car','motorcycle','airplane','bus','train','truck','boat',
    'traffic light','fire hydrant','stop sign','parking meter','bench',
    'bird','cat','dog','horse','sheep','cow','elephant','bear','zebra','giraffe',
    'backpack','umbrella','handbag','tie','suitcase','frisbee','skis','snowboard',
    'sports ball','kite','baseball bat','baseball glove','skateboard','surfboard',
    'tennis racket','bottle','wine glass','cup','fork','knife','spoon','bowl',
    'banana','apple','sandwich','orange','broccoli','carrot','hot dog','pizza',
    'donut','cake','chair','couch','potted plant','bed','dining table','toilet',
    'tv','laptop','mouse','remote','keyboard','cell phone','microwave','oven',
    'toaster','sink','refrigerator','book','clock','vase','scissors',
    'teddy bear','hair drier','toothbrush',
]
OBJECT_VOCAB = set(ALL_COCO)
for syns in COCO_SYNONYMS.values(): OBJECT_VOCAB.update(syns)
OBJECT_VOCAB.update(MULTIWORD.keys())


def find_content_words(caption, gt_objects):
    gt_norm     = {o.lower() for o in gt_objects}
    expanded_gt = set(gt_norm)
    for canonical, syns in COCO_SYNONYMS.items():
        if canonical in gt_norm: expanded_gt.update(syns)
    for alias, canonical in MULTIWORD.items():
        if canonical in gt_norm: expanded_gt.add(alias)
    doc = nlp(caption)
    obj_words, hall_words = [], []
    for tok in doc:
        w = tok.text.lower().strip()
        if tok.pos_ not in ('NOUN', 'PROPN') or len(w) < 2: continue
        canonical = MULTIWORD.get(w, w)
        if w in OBJECT_VOCAB or canonical in OBJECT_VOCAB:
            obj_words.append(w)
            if w not in expanded_gt and canonical not in expanded_gt:
                hall_words.append(w)
    return obj_words, hall_words


def score_chair(captions_gt):
    chairs_list, chairi_list = [], []
    for cap, gt in captions_gt:
        if not cap: continue
        obj_w, hall_w = find_content_words(cap, gt)
        chairs_list.append(1 if hall_w else 0)
        chairi_list.append(len(hall_w) / max(len(obj_w), 1))
    return (float(np.mean(chairs_list)) if chairs_list else 0.0,
            float(np.mean(chairi_list)) if chairi_list else 0.0)


print('CHAIR scorer ready.')

CHAIR scorer ready.


## 8. Results: Baseline vs SPIN + Bootstrap CIs (all budgets)

Runs on CPU after captions are generated. Loads budget 80 from completed artifacts; 64/128 from Drive cache.


In [13]:
# Constants (reuse from Section 5 if already run; else defaults)
try:
    SPIN_START_LAYER
except NameError:
    SPIN_START_LAYER, SPIN_END_LAYER = 0, 32
    SPIN_ROUTED_HEAD, SPIN_SMALL_NUM_MASK = 0.95, 0.08
    BUDGETS_ALL = [80, 64, 128]
    PROMPT = 'USER: <image>\nDescribe this image in detail.\nASSISTANT:'
    S4_LOCAL = LOCAL / 'stage4_400img_results.json'
    if not S4_LOCAL.exists():
        S4_URL = ('https://raw.githubusercontent.com/armaansandhu26/causal-grounding-lora/'
                  'refs/heads/master/results/stage4_400img_results.json')
        urllib.request.urlretrieve(S4_URL, str(S4_LOCAL))
    with open(S4_LOCAL) as f:
        s4_boot = json.load(f)
    eval_images = [r['img_id'] for r in s4_boot['eval_captions']]
    eval_gt_objects = [set(r['gt']) for r in s4_boot['eval_captions']]

MANIFEST_PATH = DRIVE / 'results' / 'spin_experiment_manifest.json'
if not MANIFEST_PATH.exists():
    MANIFEST_PATH = Path('results/spin_experiment_manifest.json')


def bootstrap_ci(values, n_boot=2000, seed=42):
    rng = np.random.RandomState(seed)
    arr = np.array(values)
    boot = [arr[rng.randint(0, len(arr), len(arr))].mean() for _ in range(n_boot)]
    return float(arr.mean()), float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))


def score_condition(paired, cap_key):
    chair_pairs = [(p[cap_key], p['gt']) for p in paired if p.get(cap_key)]
    chairs, chairi = score_chair(chair_pairs)
    lengths = [len(p[cap_key].split()) for p in paired if p.get(cap_key)]
    per_img_s, per_img_i = [], []
    for p in paired:
        cap = p.get(cap_key, '')
        if not cap:
            continue
        obj_w, hall_w = find_content_words(cap, p['gt'])
        per_img_s.append(1 if hall_w else 0)
        per_img_i.append(len(hall_w) / max(len(obj_w), 1))
    cs_m, cs_lo, cs_hi = bootstrap_ci(per_img_s)
    ci_m, ci_lo, ci_hi = bootstrap_ci(per_img_i)
    return {
        'CHAIRs': chairs, 'CHAIRs_CI': [cs_lo, cs_hi],
        'CHAIRi': chairi, 'CHAIRi_CI': [ci_lo, ci_hi],
        'avg_len': float(np.mean(lengths)) if lengths else 0.0,
        'n': len(chair_pairs),
    }


def load_caption_rows(path):
    p = Path(path)
    if not p.exists():
        return {}
    with open(p) as f:
        rows = json.load(f)
    return {int(r['img_id']): r['caption'] for r in rows if r.get('caption')}


gt_by_id = {eval_images[i]: eval_gt_objects[i] for i in range(len(eval_images))}
all_results = {}

for budget in BUDGETS_ALL:
    print(f'\n{"="*80}')
    print(f'BUDGET max_new_tokens = {budget}')
    print(f'{"="*80}')

    spin_path = DRIVE / 'cache' / f'spin_captions_budget{budget}.json'
    base_path = DRIVE / 'cache' / f'baseline_captions_budget{budget}.json'
    if not spin_path.exists():
        spin_path = Path(f'results/spin_captions_budget{budget}.json')
    if not base_path.exists():
        base_path = Path(f'results/baseline_captions_budget{budget}.json')

    spin_by_id = load_caption_rows(spin_path)
    base_by_id = load_caption_rows(base_path)

    paired = []
    for img_id in eval_images:
        if img_id not in spin_by_id or img_id not in base_by_id:
            continue
        paired.append({
            'img_id': img_id,
            'gt': gt_by_id[img_id],
            'baseline_cap': base_by_id[img_id],
            'spin_cap': spin_by_id[img_id],
        })

    if not paired:
        print(f'  No paired captions yet for budget={budget}. Run Section 6 first.')
        continue

    print(f'  Paired: {len(paired)}/{len(eval_images)}')
    if len(paired) < len(eval_images):
        print(f'  Warning: {len(eval_images) - len(paired)} images missing')

    budget_results = {}
    for method, cap_key in [('Baseline', 'baseline_cap'), ('SPIN (Sarkar+25)', 'spin_cap')]:
        budget_results[method] = score_condition(paired, cap_key)

    all_results[f'budget_{budget}'] = {
        'max_new_tokens': budget,
        'n_paired': len(paired),
        'spin_config': {
            'start_layer': SPIN_START_LAYER,
            'end_layer': SPIN_END_LAYER,
            'routed_head': SPIN_ROUTED_HEAD,
            'small_num_mask': SPIN_SMALL_NUM_MASK,
            'note': 'Paper Table 1 LLaVA-7B greedy CHAIR (r=0.05, alpha=0.08)',
        },
        'results': budget_results,
    }

    base_cs = budget_results['Baseline']['CHAIRs']
    print(f'\n{"Method":<22} {"CHAIRs":>8} {"95% CI":>18}  '
          f'{"CHAIRi":>8} {"95% CI":>18} {"AvgLen":>7}')
    print('-'*80)
    for method, r in budget_results.items():
        cs_lo, cs_hi = r['CHAIRs_CI']
        ci_lo, ci_hi = r['CHAIRi_CI']
        delta = '' if method == 'Baseline' else \
                f'  ({(base_cs - r["CHAIRs"]) / base_cs * 100:+.1f}%)'
        print(f'{method:<22} {r["CHAIRs"]:>8.3f} [{cs_lo:.3f}, {cs_hi:.3f}]  '
              f'{r["CHAIRi"]:>8.3f} [{ci_lo:.3f}, {ci_hi:.3f}] {r["avg_len"]:>7.1f}{delta}')
    print('-'*80)

    print('\n--- Sample outputs ---')
    for p in paired[:2]:
        print(f'  [{p["img_id"]}] GT: {sorted(p["gt"])[:4]}')
        print(f'    Baseline: {p["baseline_cap"][:100]}')
        print(f'    SPIN:     {p["spin_cap"][:100]}')
        print()



BUDGET max_new_tokens = 80
  Paired: 400/400

Method                   CHAIRs             95% CI    CHAIRi             95% CI  AvgLen
--------------------------------------------------------------------------------
Baseline                  0.365 [0.318, 0.412]     0.138 [0.116, 0.160]    60.8
SPIN (Sarkar+25)          0.367 [0.323, 0.412]     0.142 [0.121, 0.165]    58.4  (-0.7%)
--------------------------------------------------------------------------------

--- Sample outputs ---
  [293474] GT: ['book', 'fire hydrant']
    Baseline: The image features a yellow fire hydrant situated on a sidewalk next to a building. The fire hydrant
    SPIN:     The image features a yellow fire hydrant prominently placed on a sidewalk in front of a building. Th

  [465878] GT: ['person', 'surfboard']
    Baseline: The image captures a man skillfully riding a surfboard on a wave in the ocean. He is positioned in t
    SPIN:     The image captures a man skillfully riding a wave on a surfboard in the

## 9. Save results

In [14]:
out = {
    'experiment': 'SPIN vs Baseline on Stage 4 eval set (400 images)',
    'manifest': 'results/spin_experiment_manifest.json',
    'prompt': PROMPT,
    'spin_config': {
        'start_layer': SPIN_START_LAYER,
        'end_layer': SPIN_END_LAYER,
        'routed_head': SPIN_ROUTED_HEAD,
        'small_num_mask': SPIN_SMALL_NUM_MASK,
        'note': 'Paper Table 1 LLaVA-7B greedy CHAIR (r=0.05 suppressed, alpha=0.08)',
    },
}

for key, val in all_results.items():
    out[key] = val
    out[key]['status'] = 'complete' if val.get('n_paired', 0) >= 400 else 'partial'

out_path = DRIVE / 'results' / 'spin_comparison.json'
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2)
print(f'Saved: {out_path}')
print('Copy to repo: results/spin_comparison.json')


Saved: /content/drive/MyDrive/llava_hallucination_heads/results/spin_comparison.json
Copy to repo: results/spin_comparison.json
